In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

In [14]:
train_sample_path = "../train_sample.csv"
test_sample_path = "../test_sample.csv"

In [18]:
train_sample = pd.read_csv(train_sample_path)
train_sample.head(2)

,start_point,end_point,time_of_day,day_of_week,traffic_condition,event_count,is_holiday,vehicle_density,population_density,weather,public_transport_availability,historical_delay_factor,travel_time
0,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),day,Sunday,NaN,9,1,NaN,high,NaN,1,0.878909,26.907612
1,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),morning,Thursday,NaN,7,1,medium,high,NaN,1,1.081668,27.489129


In [19]:
test_sample = pd.read_csv(test_sample_path)
test_sample.head(2)

,start_point,end_point,time_of_day,day_of_week,traffic_condition,event_count,is_holiday,vehicle_density,population_density,weather,public_transport_availability,historical_delay_factor
0,West Jakarta (Jakarta Barat),East Jakarta (Jakarta Timur),morning,Saturday,5.0,8,1,medium,NaN,NaN,2,1.126429
1,South Jakarta (Jakarta Selatan),East Jakarta (Jakarta Timur),evening,Saturday,NaN,9,1,low,medium,fog,2,1.121015


In [20]:
cols_to_drop = ['weather']
train_sample = train_sample.drop(columns=cols_to_drop)

test_sample = test_sample.drop(columns=cols_to_drop)

In [21]:
train_sample = train_sample.dropna()
train_sample.isnull().sum()

start_point                      0
end_point                        0
time_of_day                      0
day_of_week                      0
traffic_condition                0
event_count                      0
is_holiday                       0
vehicle_density                  0
population_density               0
public_transport_availability    0
historical_delay_factor          0
travel_time                      0
dtype: int64

In [22]:
# Numeric → mean
numeric_cols = test_sample.select_dtypes(include='number').columns

test_sample[numeric_cols] = test_sample[numeric_cols].fillna(
    test_sample[numeric_cols].mean()
)

# Categorical → mode
categorical_cols = test_sample.select_dtypes(
    include=['str']
).columns

for col in categorical_cols:
    test_sample[col] = test_sample[col].fillna(
        test_sample[col].mode()[0]
    )

test_sample.isnull().sum()
test_sample.isnull().sum()

start_point                      0
end_point                        0
time_of_day                      0
day_of_week                      0
traffic_condition                0
event_count                      0
is_holiday                       0
vehicle_density                  0
population_density               0
public_transport_availability    0
historical_delay_factor          0
dtype: int64

In [23]:
train_sample.info()

<class 'pandas.DataFrame'>
Index: 10450 entries, 3 to 39997
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   start_point                    10450 non-null  str    
 1   end_point                      10450 non-null  str    
 2   time_of_day                    10450 non-null  str    
 3   day_of_week                    10450 non-null  str    
 4   traffic_condition              10450 non-null  float64
 5   event_count                    10450 non-null  int64  
 6   is_holiday                     10450 non-null  int64  
 7   vehicle_density                10450 non-null  str    
 8   population_density             10450 non-null  str    
 9   public_transport_availability  10450 non-null  int64  
 10  historical_delay_factor        10450 non-null  float64
 11  travel_time                    10450 non-null  float64
dtypes: float64(3), int64(3), str(6)
memory usage: 1.0 MB


In [24]:
test_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   start_point                    3000 non-null   str    
 1   end_point                      3000 non-null   str    
 2   time_of_day                    3000 non-null   str    
 3   day_of_week                    3000 non-null   str    
 4   traffic_condition              3000 non-null   float64
 5   event_count                    3000 non-null   int64  
 6   is_holiday                     3000 non-null   int64  
 7   vehicle_density                3000 non-null   str    
 8   population_density             3000 non-null   str    
 9   public_transport_availability  3000 non-null   int64  
 10  historical_delay_factor        3000 non-null   float64
dtypes: float64(2), int64(3), str(6)
memory usage: 257.9 KB


train val split

In [25]:
X_train, X_val, y_train, y_val = train_test_split(train_sample.drop(columns=['travel_time']), train_sample['travel_time'], test_size=0.2, random_state=38)

In [26]:
X_test = test_sample

In [27]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_train index:", X_train.index[:5])
print("y_train index:", y_train.index[:5])

X_train: (8360, 11)
y_train: (8360,)
X_train index: Index([6368, 27162, 16780, 17158, 32067], dtype='int64')
y_train index: Index([6368, 27162, 16780, 17158, 32067], dtype='int64')


one-hot encoding

In [ ]:
# ohe = OneHotEncoder(sparse=False, drop='first', handle_unknown='ignore')
# X_train_encoded = ohe.fit_transform(X_train)
# X_val_encoded = ohe.transform(X_val)
# X_test_encoded = ohe.transform(test_sample)
# X_train_encoded.head(3)

In [28]:
categorical_cols = train_sample.select_dtypes(include=['str', 'object']).columns

In [29]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error']
)

print('R²:', scores['test_r2'].mean())
print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
print('MSE:', -scores['test_neg_mean_squared_error'].mean())

R²: 0.7235287228240531
MAE: 5.951174968305675
MSE: 63.47750780752508


In [30]:
model.fit(X_train, y_train)

y_pred = model.predict(test_sample)

pd.DataFrame(y_pred).to_csv('submission.csv', index=False)

In [31]:
pd.read_csv('submission.csv')['0']

0       45.544539
1       21.229395
2       29.408217
3       21.001665
4       37.125164
          ...    
2995    35.307568
2996    30.357265
2997    17.019444
2998    24.978245
2999    27.300790
Name: 0, Length: 3000, dtype: float64